###### Обучение нейросети #

In [ ]:
import source.main
import pandas as pd
import matplotlib.pyplot as plt

# 2. Просто вызываем функцию, которая обучит глобальный app внутри main.py
n = source.main.learn_and_validate()

# 3. Читаем данные
file_pred = pd.read_csv("data/submission_template.csv")
all_match_id = file_pred['match_id'].unique().tolist()

file_name = pd.read_csv("data/matches.csv")
op1 = file_name['opponent'].tolist()
op2 = file_name['opponent_khl'].tolist()

# 4. Словарь
match_names = {}
for mid in all_match_id:
    try:
        num = int(''.join(filter(str.isdigit, str(mid))))
        idx = num - 1
        match_names[mid] = f"{op1[idx]} vs {op2[idx]}" if 0 <= idx < len(op1) else f"Unknown_{mid}"
    except Exception:
        match_names[mid] = f"Unknown_{mid}"

# 5. Все 8 зон
ZONES = list(range(1, 9))

# 6. Считаем pred, обращаясь к source.main.app
results = []
for match in all_match_id:
    name = match_names.get(match, f"match_{match}")
    for zone in ZONES:
        try:
            # !!! Используем source.main.app !!!
            pred = source.main.app.get_game_zone_prediction(str(match), str(zone))
            results.append({
                'match_id': match,
                'match_name': name,
                'zone': zone,
                'prediction': pred
            })
        except Exception as e:
            print(f"Пропущено: {match} / {zone} — {e}")


###### Построение диаграмм #

In [ ]:

df_results = pd.DataFrame(results)

# 6. Матрица: индекс — УНИКАЛЬНЫЙ ключ (match_id + имя),
#    чтобы 17 матчей не склеились в 14
df_results['key'] = df_results['match_id'] + " | " + df_results['match_name']

matrix_pred = df_results.pivot_table(
    index='key',
    columns='zone',
    values='prediction',
    aggfunc='first'
)


# 7. Диаграммы — чистое имя в заголовке
for key, row in matrix_pred.iterrows():
    match_id, clean_name = key.split(" | ", 1)   # отделяем id от имени

    fig, ax = plt.subplots(figsize=(8, 4))

    zones  = row.index.tolist()
    values = row.values

    bars = ax.bar(zones, values, color='steelblue', edgecolor='black')
    for bar, v in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{int(v)}",
            ha='center', va='bottom', fontsize=8
        )

    ax.set_title(clean_name, fontsize=11, fontweight='bold')  # <- чистое имя
    ax.set_xlabel("Зона")
    ax.set_ylabel("Билеты")
    ax.set_xticks(zones)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()



###### Построение таблицы #

In [ ]:
source.main.form_predictions_table()
pd.set_option('display.max_rows', None)
df_predictions = pd.read_csv("predictions.csv")
df_predictions